# MIMIC-IV data-processing and table-join pipeline

This notebook demonstrates how MIMIC-IV tables are joined into one admission-level, analysis-ready source table. Source identifiers are used **only for local linkage** and are removed before synthetic data are exported. Never export the intermediate real patient-level table.

Join flow:

```text
patients (1 row/subject)
        │ inner join on subject_id
        ▼
admissions (1 row/hadm_id) ── inner join ── diagnosis summary (1 row/hadm_id)
        │                                      from diagnoses_icd
        ├── left join ICU indicator (1 row/hadm_id after aggregation)
        │                                      from icustays
        └── left join early labs (1 row/hadm_id after pivot)
                                               from labevents + d_labitems
```

Why these join types? Demographics and diagnosis are required cohort elements, so they use inner joins. ICU and individual labs are optional features, so they use left joins to avoid excluding non-ICU admissions or patients with missing tests.

In [ ]:
from pathlib import Path
import logging
import numpy as np
import pandas as pd

import os
# Point this at YOUR credentialed local copy of MIMIC-IV (PhysioNet). Never commit it.
MIMIC_ROOT = Path(os.environ.get('MIMIC_IV_ROOT', 'path/to/mimic-iv-2.2'))
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUT_ROOT = REPO_ROOT / 'data' / 'profiles_1000'
EARLY_WINDOW_HOURS = 24
MAX_LOS_DAYS = 60
LAB_CHUNKSIZE = 1_000_000
RANDOM_SEED = 42

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
log = logging.getLogger('mimic-processing')

## 1. Discover the actual installation

The supplied directory contains an additional version directory on this machine. This function resolves it recursively and prefers `.csv.gz` over duplicate uncompressed CSV files.

In [ ]:
def resolve_mimic_root(root: Path) -> Path:
    root = root.expanduser().resolve()
    candidates = [root] + [p.parent for p in root.rglob('hosp') if p.is_dir()]
    for candidate in candidates:
        if (candidate / 'hosp').is_dir():
            return candidate
    raise FileNotFoundError(f'No MIMIC-IV hosp directory found beneath {root}')

def locate_table(base: Path, module: str, table: str, required: bool = True) -> Path | None:
    folder = base / module
    candidates = [folder / f'{table}.csv.gz', folder / f'{table}.csv', folder / f'{table}.parquet']
    path = next((p for p in candidates if p.is_file()), None)
    if required and path is None:
        raise FileNotFoundError(f'Required table missing: {module}/{table}; checked {candidates}')
    return path

def read_table(path: Path, usecols=None, **kwargs) -> pd.DataFrame:
    if path.suffix == '.parquet':
        return pd.read_parquet(path, columns=usecols)
    return pd.read_csv(path, usecols=usecols, low_memory=False, **kwargs)

base = resolve_mimic_root(MIMIC_ROOT)
tables = {
    'patients': locate_table(base, 'hosp', 'patients'),
    'admissions': locate_table(base, 'hosp', 'admissions'),
    'diagnoses_icd': locate_table(base, 'hosp', 'diagnoses_icd'),
    'd_labitems': locate_table(base, 'hosp', 'd_labitems', required=False),
    'labevents': locate_table(base, 'hosp', 'labevents', required=False),
    'icustays': locate_table(base, 'icu', 'icustays', required=False),
}
print('Resolved root:', base)
for name, path in tables.items():
    if path is not None:
        print(name, path.name, 'columns=', list(read_table(path, nrows=0).columns) if path.suffix != '.parquet' else list(pd.read_parquet(path).columns))
    else:
        print(f'{name:16s} OPTIONAL TABLE NOT INSTALLED')

## 2. Inner join admissions to patients

`admissions` is many-to-one relative to `patients`. `validate='many_to_one'` makes pandas fail if that assumption is violated. Age is reconstructed at admission using anchor age and anchor year; exact dates will not enter the final feature table.

In [ ]:
patients = read_table(tables['patients'], ['subject_id', 'gender', 'anchor_age', 'anchor_year'])
admissions = read_table(
    tables['admissions'],
    ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type',
     'admission_location', 'insurance', 'marital_status', 'race', 'hospital_expire_flag'],
)
admissions['admittime'] = pd.to_datetime(admissions['admittime'], errors='coerce')
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'], errors='coerce')

flow = [('initial admissions', len(admissions))]
cohort = admissions.merge(
    patients, on='subject_id', how='inner', validate='many_to_one', indicator='_patient_join'
)
assert cohort['_patient_join'].eq('both').all()
cohort = cohort.drop(columns='_patient_join')
cohort['age'] = cohort['anchor_age'] + cohort['admittime'].dt.year - cohort['anchor_year']
cohort['length_of_stay_days'] = (cohort['dischtime'] - cohort['admittime']).dt.total_seconds() / 86_400
cohort = cohort[cohort['age'].ge(18)].copy()
flow.append(('adult admissions', len(cohort)))
cohort = cohort[cohort['length_of_stay_days'].gt(0) & cohort['length_of_stay_days'].le(MAX_LOS_DAYS)].copy()
flow.append(('valid length of stay', len(cohort)))
print('Rows after patients ⨝ admissions:', len(cohort))
cohort[['subject_id', 'hadm_id', 'age', 'length_of_stay_days']].head()

## 3. Aggregate diagnoses, then inner join

Joining raw diagnoses directly would create multiple rows per admission. First, choose the principal code using the smallest `seq_num`, calculate the disease class, and aggregate comorbidity flags to one row per `hadm_id`. Then enforce a one-to-one inner join.

In [ ]:
def broad_disease_class(code: object, version: object) -> str:
    code = str(code).upper().replace('.', '')
    if int(version) == 10:
        prefix_map = {'A':'Infectious','B':'Infectious','C':'Neoplasm','E':'Endocrine_or_metabolic',
                      'F':'Psychiatric','G':'Neurologic','I':'Cardiovascular','J':'Respiratory',
                      'K':'Gastrointestinal_or_liver','M':'Musculoskeletal','N':'Kidney_or_urinary',
                      'S':'Injury_or_poisoning','T':'Injury_or_poisoning'}
        if code.startswith('D'):
            return 'Neoplasm' if len(code) > 1 and code[1] in '01234' else 'Hematologic'
        return prefix_map.get(code[:1], 'Other')
    try:
        number = int(code[:3])
    except ValueError:
        return 'Other'
    ranges = [(1,139,'Infectious'),(140,239,'Neoplasm'),(240,279,'Endocrine_or_metabolic'),
              (280,289,'Hematologic'),(290,319,'Psychiatric'),(320,389,'Neurologic'),
              (390,459,'Cardiovascular'),(460,519,'Respiratory'),
              (520,579,'Gastrointestinal_or_liver'),(580,629,'Kidney_or_urinary'),
              (710,739,'Musculoskeletal'),(800,999,'Injury_or_poisoning')]
    return next((label for low, high, label in ranges if low <= number <= high), 'Other')

COMORBIDITY_PREFIXES = {
    'has_diabetes': {9:['250'], 10:['E10','E11','E12','E13','E14']},
    'has_hypertension': {9:['401','402','403','404','405'], 10:['I10','I11','I12','I13','I15']},
    'has_ckd': {9:['585'], 10:['N18']},
    'has_heart_failure': {9:['428'], 10:['I50']},
    'has_copd': {9:['490','491','492','496'], 10:['J40','J41','J42','J43','J44']},
    'has_liver_disease': {9:['570','571','572','573'], 10:['K70','K71','K72','K73','K74','K75','K76']},
    'has_cancer': {9:[str(i) for i in range(140, 209)], 10:['C']},
    'has_sepsis': {9:['038','99591','99592'], 10:['A40','A41']},
    'has_anemia': {9:['280','281','282','283','284','285'], 10:['D50','D51','D52','D53','D54','D55','D56','D57','D58','D59','D60','D61','D62','D63','D64']},
}

diagnoses = read_table(tables['diagnoses_icd'], ['hadm_id', 'seq_num', 'icd_code', 'icd_version'])
diagnoses = diagnoses[diagnoses['hadm_id'].isin(cohort['hadm_id'])].copy()
diagnoses['seq_num'] = pd.to_numeric(diagnoses['seq_num'], errors='coerce')
diagnoses['normalized_code'] = diagnoses['icd_code'].astype(str).str.upper().str.replace('.', '', regex=False)
diagnoses['primary_disease_class'] = [broad_disease_class(c, v) for c, v in zip(diagnoses.icd_code, diagnoses.icd_version)]
principal = (diagnoses.sort_values(['hadm_id','seq_num']).drop_duplicates('hadm_id')
             [['hadm_id','primary_disease_class']])
for flag, by_version in COMORBIDITY_PREFIXES.items():
    diagnoses[flag] = [int(any(code.startswith(p) for p in by_version[int(version)]))
                       for code, version in zip(diagnoses.normalized_code, diagnoses.icd_version)]
flags = list(COMORBIDITY_PREFIXES)
diagnosis_summary = principal.merge(diagnoses.groupby('hadm_id', as_index=False)[flags].max(),
                                    on='hadm_id', how='inner', validate='one_to_one')
assert diagnosis_summary['hadm_id'].is_unique
cohort = cohort.merge(diagnosis_summary, on='hadm_id', how='inner', validate='one_to_one')
flow.append(('with diagnosis summary', len(cohort)))
print('Rows after inner joining diagnosis summary:', len(cohort))

## 4. Left join ICU utilization

An admission can contain multiple ICU stays. Aggregate first with `max`, producing a single binary admission-level indicator. A left join preserves admissions without ICU stays.

In [ ]:
if tables['icustays'] is not None:
    icu = read_table(tables['icustays'], ['hadm_id'])
    icu = icu[icu['hadm_id'].isin(cohort['hadm_id'])].drop_duplicates('hadm_id')
    icu['icu_admission'] = 1
    cohort = cohort.merge(icu, on='hadm_id', how='left', validate='one_to_one')
    cohort['icu_admission'] = cohort['icu_admission'].fillna(0).astype(int)
else:
    cohort['icu_admission'] = 0
print(cohort['icu_admission'].value_counts(dropna=False))

## 5. Chunk, aggregate, pivot, and left join early labs

`labevents` is too large to load in full. Each chunk is filtered immediately to eligible admissions, selected item IDs, and chart times from admission through 24 hours. We retain the first valid value for each admission/test and pivot from long to wide before joining. Clinical bounds prevent extreme values; production code should additionally audit unit distributions before accepting a site-specific item ID.

In [ ]:
LAB_SPECS = {
 'hemoglobin': (['Hemoglobin'], 2, 25),
 'white_blood_cell_count': (['White Blood Cells'], .1, 200),
 'platelet_count': (['Platelet Count'], 1, 2000),
 'sodium': (['Sodium'], 90, 200), 'potassium': (['Potassium'], 1, 10),
 'chloride': (['Chloride'], 50, 160), 'bicarbonate': (['Bicarbonate'], 2, 60),
 'blood_urea_nitrogen': (['Urea Nitrogen'], 1, 300),
 'creatinine': (['Creatinine'], .1, 30), 'glucose': (['Glucose'], 10, 1500),
 'calcium': (['Calcium','Total Calcium'], 2, 20), 'albumin': (['Albumin'], .5, 8),
 'bilirubin': (['Bilirubin, Total'], 0, 80),
 'ast': (['Asparate Aminotransferase (AST)'], 1, 20000),
 'alt': (['Alanine Aminotransferase (ALT)'], 1, 20000),
 'inr': (['INR(PT)'], .5, 20), 'lactate': (['Lactate'], .1, 40),
}

def extract_early_labs(cohort: pd.DataFrame) -> pd.DataFrame:
    if tables['d_labitems'] is None or tables['labevents'] is None:
        log.warning('Lab tables unavailable; returning admissions without lab columns')
        return pd.DataFrame({'hadm_id': cohort['hadm_id']})
    items = read_table(tables['d_labitems'], ['itemid','label'])
    item_rows = []
    for feature, (labels, low, high) in LAB_SPECS.items():
        hits = items[items['label'].str.lower().isin({x.lower() for x in labels})]
        item_rows.extend({'itemid': r.itemid, 'feature': feature, 'low': low, 'high': high}
                         for r in hits.itertuples())
    item_map = pd.DataFrame(item_rows).drop_duplicates('itemid')
    eligible_ids = set(cohort['hadm_id'])
    admission_times = cohort.set_index('hadm_id')['admittime']
    pieces = []
    reader = pd.read_csv(tables['labevents'],
                         usecols=['hadm_id','itemid','charttime','valuenum','valueuom'],
                         chunksize=LAB_CHUNKSIZE, low_memory=False)
    for chunk_number, chunk in enumerate(reader, 1):
        chunk = chunk[chunk['hadm_id'].isin(eligible_ids) &
                      chunk['itemid'].isin(set(item_map['itemid'])) & chunk['valuenum'].notna()].copy()
        if not chunk.empty:
            chunk['charttime'] = pd.to_datetime(chunk['charttime'], errors='coerce')
            chunk['admittime'] = chunk['hadm_id'].map(admission_times)
            hours = (chunk['charttime'] - chunk['admittime']).dt.total_seconds() / 3600
            pieces.append(chunk[hours.between(0, EARLY_WINDOW_HOURS)])
        if chunk_number % 5 == 0:
            log.info('Scanned approximately %s lab rows', f'{chunk_number * LAB_CHUNKSIZE:,}')
    if not pieces:
        return pd.DataFrame({'hadm_id': cohort['hadm_id']})
    lab_long = pd.concat(pieces, ignore_index=True).merge(item_map, on='itemid', how='inner')
    lab_long = lab_long[lab_long['valuenum'].between(lab_long['low'], lab_long['high'])]
    first = (lab_long.sort_values('charttime').drop_duplicates(['hadm_id','feature'])
             [['hadm_id','feature','valuenum']])
    return first.pivot(index='hadm_id', columns='feature', values='valuenum').reset_index()

early_labs = extract_early_labs(cohort)
assert early_labs['hadm_id'].is_unique
cohort = cohort.merge(early_labs, on='hadm_id', how='left', validate='one_to_one')
print('Rows after optional lab left join:', len(cohort))

## 6. Select one admission per patient and engineer analysis features

Keeping the earliest qualifying admission yields unique source patients. Exact timestamps and discharge fields are converted to safe aggregate features or targets and then removed. This table is suitable for fitting the synthetic generator locally, but it remains real deidentified MIMIC data and must not be exported.

In [ ]:
cohort = cohort.sort_values('admittime').drop_duplicates('subject_id', keep='first').copy()
flow.append(('one admission per patient', len(cohort)))
cohort['sex'] = cohort['gender'].map({'M':'Male','F':'Female'}).fillna('Unknown')
cohort['age'] = cohort['age'].clip(18, 100).round()
cohort['age_group'] = pd.cut(cohort['age'], [17,39,59,79,120],
                             labels=['18-39','40-59','60-79','80+']).astype(str)
cohort['race_group'] = cohort['race'].fillna('Unknown').str.upper().map(
    lambda s: next((x for x in ['WHITE','BLACK','ASIAN','HISPANIC'] if x in s), 'OTHER_OR_UNKNOWN').title())
cohort['emergency_admission'] = cohort['admission_type'].str.contains('EMERGENCY|URGENT', case=False).astype(int)
cohort['weekday_admission'] = cohort['admittime'].dt.dayofweek.lt(5).astype(int)
cohort['hour_group'] = pd.cut(cohort['admittime'].dt.hour, [-1,5,11,17,23],
                              labels=['Night','Morning','Afternoon','Evening']).astype(str)
cohort['in_hospital_mortality'] = cohort['hospital_expire_flag'].astype(int)
cohort['comorbidity_count'] = cohort[list(COMORBIDITY_PREFIXES)].sum(axis=1)
cohort['log1p_length_of_stay'] = np.log1p(cohort['length_of_stay_days'])

safe_features = [
 'age','age_group','sex','race_group','marital_status','insurance','admission_type',
 'admission_location','emergency_admission','weekday_admission','hour_group','icu_admission',
 'primary_disease_class','comorbidity_count',*COMORBIDITY_PREFIXES.keys(),*LAB_SPECS.keys(),
 'in_hospital_mortality','length_of_stay_days','log1p_length_of_stay'
]
safe_features = [c for c in safe_features if c in cohort.columns]
analysis_ready_source = cohort[safe_features].copy()
assert not {'subject_id','hadm_id','stay_id','admittime','dischtime'} & set(analysis_ready_source.columns)
print('Analysis-ready source shape:', analysis_ready_source.shape)
pd.DataFrame(flow, columns=['stage','n_admissions'])

## 7. Leakage-specific modeling tables

The synthetic profiles already produced in `D:\CTSA` are safe to load for workshop analysis. Separate feature sets exclude targets and information that may only become known later in the admission.

In [ ]:
synthetic = pd.read_csv(OUTPUT_ROOT / 'synthetic_patient_profiles_1000.csv')
assert len(synthetic) == 1000 and synthetic['synthetic_patient_id'].is_unique

disease_dataset = pd.read_csv(OUTPUT_ROOT / 'disease_classification_dataset.csv')
mortality_dataset = pd.read_csv(OUTPUT_ROOT / 'mortality_prediction_dataset.csv')
los_dataset = pd.read_csv(OUTPUT_ROOT / 'length_of_stay_regression_dataset.csv')

assert 'primary_disease_class' in disease_dataset
assert 'length_of_stay_days' not in mortality_dataset
assert 'primary_disease_class' not in mortality_dataset
assert 'in_hospital_mortality' not in los_dataset
assert 'log1p_length_of_stay' not in los_dataset

pd.DataFrame({
    'table': ['full synthetic profiles','disease classification','mortality classification','LOS regression'],
    'rows': [len(synthetic),len(disease_dataset),len(mortality_dataset),len(los_dataset)],
    'columns': [synthetic.shape[1],disease_dataset.shape[1],mortality_dataset.shape[1],los_dataset.shape[1]],
    'target': ['multiple','primary_disease_class','in_hospital_mortality','length_of_stay_days'],
})

## Privacy and use notice

Do not save or display `analysis_ready_source` outside the authorized local environment. Only the already validated synthetic tables in `D:\CTSA` are intended for workshop use. Synthetic data are for education and research only, are not clinically validated, and do not replace compliance with the MIMIC-IV data-use agreement.